# Altı ölçekte göreli doku ve üç komşu
Doku ölçülerinin toplam büyüklüğünü kaldırmanın komşu seçimine etkisini iki yerel sınavda ölçüyoruz.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.preprocessing import StandardScaler

root = Path("/kaggle/input") if Path("/kaggle/input").exists() else next(p for p in [Path("work/soil-data"), Path("../work/soil-data")] if p.exists())
data = next(root.rglob("Training_labels_updated.csv")).parent
labels = pd.read_csv(data / "Training_labels_updated.csv").set_index("sample_id")
cameras = pd.read_csv(data / "ppm_updated.csv")
photos = sorted(data.joinpath("Training-All_Photos_updated").rglob("*.jpg"))
print(f"{len(labels)} numune, {len(photos)} eğitim fotoğrafı")

24 numune, 127 eğitim fotoğrafı


## 1. Fotoğrafı bir avuç sayıyla anlatmak
Bütün fotoğrafların merkezinden yaklaşık 100 × 100 mm alan alıyoruz. Kamera tablosundaki piksel/mm değerini gerçek dosyanın çözünürlüğüne uyarlıyoruz. Bu kalibrasyon, çekim uzaklığının kamera tablosunda varsayılana yakın olduğunu kabul eder.

Bu alanı griye çevirip 400 × 400 piksele getiriyoruz. Yatay ve dikey yönde, birbirinden 0,25 mm ile 8 mm arasında uzakta olan noktaların ne kadar farklı aydınlıkta göründüğünü ölçüyoruz. Böylece her fotoğrafta aynı altı fiziksel uzaklığı karşılaştırıyoruz. Her numunenin fotoğraflarını, her ölçünün ortanca değerini alarak birleştiriyoruz.

In [2]:
def physical_patch(path, camera):
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        w, h = image.size
        pixels_per_mm = camera.ppm * max(w, h) / max(camera.width, camera.height)
        side = round(100 * pixels_per_mm)
        assert 0 < side <= min(w, h)
        x, y = (w - side) // 2, (h - side) // 2
        return image.crop((x, y, x + side, y + side)).convert("RGB").resize((400, 400), Image.Resampling.LANCZOS)

def texture(patch):
    gray = np.asarray(patch.convert("L"), dtype=float)
    low, high = np.percentile(gray, [5, 95])
    gray = np.clip((gray - low) / max(high - low, 1.0), 0, 1)
    values = []
    for offset in [1, 2, 4, 8, 16, 32]:
        horizontal = np.abs(gray[:, offset:] - gray[:, :-offset]).mean()
        vertical = np.abs(gray[offset:, :] - gray[:-offset, :]).mean()
        values.append((horizontal + vertical) / 2)
    return values

camera_by_name = {c.phone.replace(" ", "_").lower(): c for c in cameras.itertuples()}
rows = []
examples = {}
for path in photos:
    phone, sample, number = path.stem.rsplit("_", 2)
    patch = physical_patch(path, camera_by_name[phone.lower()])
    rows.append([sample, *texture(patch)])
    examples.setdefault(sample, patch)
features = pd.DataFrame(rows, columns=["sample_id", "0.25", "0.5", "1", "2", "4", "8"])
features = features.groupby("sample_id").median()
assert set(features.index) == set(labels.index)
features = features.loc[labels.index]

## 2. Gerçek cevabı saklayıp tahmin yapmak
Her turda yalnızca geride kalan 23 numuneden ölçekleme değerlerini öğreniyoruz. Altı doku ölçüsüne göre uzaklık hesaplayıp en yakın üç komşuyu seçiyoruz. Komşuların ölçümlerinin ortalaması tahminimiz oluyor. Karşılaştırma için aynı 23 numunenin tamamının ortalamasını da hesaplıyoruz.

In [3]:
sizes = labels.columns.astype(float).to_numpy()
widths = np.diff(np.log10(sizes))

def emd(observed, predicted):
    return np.abs(observed[:-1] - predicted[:-1]) @ widths

scores, neighbors, predictions, averages = [], {}, {}, {}
for sample in labels.index:
    train = features.drop(index=sample)
    scaler = StandardScaler().fit(train)
    distance = np.linalg.norm(scaler.transform(train) - scaler.transform(features.loc[[sample]]), axis=1)
    nearest = train.index[np.argsort(distance, kind="stable")[:3]].tolist()
    predicted = labels.loc[nearest].mean().to_numpy()
    baseline = labels.loc[train.index].mean().to_numpy()
    observed = labels.loc[sample].to_numpy()
    assert sample not in nearest and len(set(nearest)) == 3
    assert np.all(np.diff(predicted) >= -1e-10) and np.isclose(predicted[-1], 100)
    scores.append((sample, emd(observed, baseline), emd(observed, predicted)))
    neighbors[sample], predictions[sample], averages[sample] = nearest, predicted, baseline
scores = pd.DataFrame(scores, columns=["Numune", "Ortalama", "Üç komşu"]).set_index("Numune")
print(scores.mean().round(2).to_string())
print(f"Üç komşunun daha az hata yaptığı numune sayısı: {(scores['Üç komşu'] < scores['Ortalama']).sum()}/{len(scores)}")

Ortalama    89.89
Üç komşu    44.74
Üç komşunun daha az hata yaptığı numune sayısı: 18/24


## Her çekimde ölçüleri kendi toplamına bölme

Önceki deneydeki görüntü kırpma ve gri tonları düzeltme adımlarını kullanıyoruz. Bu kez, örneğin 1 mm aralığındaki açık-koyu farkını, aynı çekimdeki altı farkın toplamına bölüyoruz. Altı ölçü birden aynı katsayıyla büyürse oranlar değişmez. Tümü sıfır olan bir fotoğrafın göreli özellikleri de sıfırdır. Her toprağın fotoğraflarını bundan sonra ortanca ile birleştiriyoruz. Başka ayar seçmiyoruz.

In [4]:
scales = features.columns.tolist()
frame = pd.DataFrame(rows, columns=["sample_id", *scales])
relative = ["pay_" + s for s in scales]
total = frame[scales].sum(axis=1)
shares = frame[scales].div(total.where(total > 0, 1), axis=0)
assert np.allclose(shares.sum(axis=1), (total > 0).astype(float))
frame[relative] = shares.to_numpy()
columns = scales + relative
soil = frame.groupby("sample_id")[relative].median().loc[labels.index]
err = []
for sample in labels.index:
    train = soil.drop(index=sample)
    scaler = StandardScaler().fit(train)
    distance = np.linalg.norm(scaler.transform(train) - scaler.transform(soil.loc[[sample]]), axis=1)
    selected = train.index[np.argsort(distance, kind="stable")[:3]]
    assert len(set(selected)) == 3 and sample not in selected
    curve = labels.loc[selected].mean().to_numpy()
    err.append(emd(labels.loc[sample].to_numpy(), curve))
comparison = scores[["Üç komşu"]].rename(columns={"Üç komşu":"Sade"}).copy()
comparison["Göreli"] = err
comparison["Kazanç"] = comparison["Sade"] - comparison["Göreli"]
assert len(comparison) == 24 and np.isfinite(comparison).all().all()
assert round(comparison["Sade"].mean(), 3) == 44.744
print(comparison.round(2).to_string())
print(comparison.mean().round(3))
print("İyileşen / aynı / kötüleşen:", (comparison.Kazanç > 1e-8).sum(), (comparison.Kazanç.abs() <= 1e-8).sum(), (comparison.Kazanç < -1e-8).sum())

          Sade  Göreli  Kazanç
Numune                        
F827     63.98   62.26    1.72
G190     24.85   35.31  -10.46
H030     44.43   27.39   17.03
H037     60.27   60.27   -0.00
H038     93.28  112.90  -19.62
H126     14.28   26.55  -12.27
H181     20.89   20.89   -0.00
H183     28.41   21.30    7.11
H366      8.11   14.72   -6.61
H367      7.67   17.29   -9.62
H368     14.50   14.50    0.00
H371     14.03   14.03    0.00
H372      7.27   10.12   -2.85
H374    128.87   92.83   36.04
H405    133.53   89.12   44.41
H493     27.70   26.31    1.39
H516     90.35   54.18   36.17
H549     13.97   36.60  -22.63
H615     11.57   12.23   -0.66
H616     85.30   89.19   -3.89
H617     71.29   66.31    4.97
H637     40.53   42.41   -1.88
H666     35.08   35.08    0.00
H668     33.70   63.82  -30.12
Sade      44.744
Göreli    43.567
Kazanç     1.177
dtype: float64
İyileşen / aynı / kötüleşen: 8 5 11


## Telefon değişiminde aynı kontrol

Her iki telefonla çekilen aynı 21 toprakta, eğitimde kullandığımız 20 toprağın çekimlerini tek telefondan alıyoruz. Saklanan toprağın iki telefondaki çekimlerini ayrı ayrı tahmin ediyoruz.

In [5]:
phones = ["Motorola_Edge", "Samsung_A52"]
frame["telefon"] = [p.stem.rsplit("_", 2)[0] for p in photos]
assert frame["sample_id"].tolist() == [p.stem.rsplit("_", 2)[1] for p in photos]
per_phone = frame.groupby(["telefon", "sample_id"])[columns].median()
motorola, samsung = [per_phone.loc[p].index for p in phones]
paired = labels.index[labels.index.isin(motorola) & labels.index.isin(samsung)]
assert len(paired) == 21 and paired.is_unique
widths = np.diff(np.log10(labels.columns.astype(float)))

def emd(a, b):
    return np.abs(a - b)[:-1] @ widths

records = []
for name, keys in [("Sade", scales), ("Göreli", relative)]:
    for source, other in [phones, phones[::-1]]:
        for sample in paired:
            train = per_phone.loc[source, keys].loc[paired.drop(sample)]
            assert len(train) == 20 and sample not in train.index
            scaler = StandardScaler().fit(train)
            reference = scaler.transform(train)
            curves = []
            for viewed in [source, other]:
                query = per_phone.loc[viewed, keys].loc[[sample]]
                distance = np.linalg.norm(reference - scaler.transform(query), axis=1)
                nearest = train.index[np.argsort(distance, kind="stable")[:3]]
                curves.append(labels.loc[nearest].mean().to_numpy())
            truth = labels.loc[sample].to_numpy()
            same_error, cross_error = [emd(curve, truth) for curve in curves]
            records.append([name, source, sample, same_error, cross_error, cross_error - same_error, emd(*curves)])
results = pd.DataFrame(records, columns=["Model", "Eğitim telefonu", "Numune", "Aynı", "Diğer", "Hata farkı", "Tahminler arası EMD"])
assert len(results) == 84 and not results.duplicated(["Model", "Eğitim telefonu", "Numune"]).any()
metrics = ["Aynı", "Diğer", "Hata farkı", "Tahminler arası EMD"]
summary = results.groupby(["Model","Eğitim telefonu"])[metrics].mean()
assert np.allclose(summary.loc["Sade", ["Aynı", "Diğer"]], [[45.98,49.10],[45.01,45.26]], atol=0.005, rtol=0)
print(summary.round(3).to_string())
print("İki eğitim yönünün eşit ağırlıklı ortalaması")
print(results.groupby("Model")[metrics].mean().round(3).to_string())

                          Aynı   Diğer  Hata farkı  Tahminler arası EMD
Model  Eğitim telefonu                                                 
Göreli Motorola_Edge    44.494  47.026       2.532               19.240
       Samsung_A52      38.542  40.298       1.756               27.841
Sade   Motorola_Edge    45.985  49.095       3.111               19.912
       Samsung_A52      45.005  45.265       0.259               27.548
İki eğitim yönünün eşit ağırlıklı ortalaması
          Aynı   Diğer  Hata farkı  Tahminler arası EMD
Model                                                  
Göreli  41.518  43.662       2.144               23.541
Sade    45.495  47.180       1.685               23.730


## On test toprağı için gönderim dosyası

Eğitimdeki 24 toprağı referans alıyoruz. Her test çekiminde altı doku ölçüsünü kendi toplamına bölüyor, ardından aynı toprağın çekimleri arasında ortanca alıyoruz.

In [6]:
import re
template = pd.read_csv(data / "sample_submission.csv").set_index("sample_id")
assert template.index.is_unique and template.columns.equals(labels.columns)
camera_map = {c.phone.replace(" ", "").lower(): c for c in cameras.itertuples()}
test_rows = []
for path in sorted((data / "Test_All_Photos").rglob("*")):
    if path.suffix.lower() != ".jpg":
        continue
    phone, suffix = path.stem.split("_HPC_", 1)
    sample = "HPC_" + re.sub(r" \(\d+\)$", "", suffix)
    sample = sample.replace("Münster_BS6_9,0-10m", "Muenster_BS6_9_0-10m")
    assert sample in template.index
    values = np.array(texture(physical_patch(path, camera_map[phone.replace("_", "").lower()])))
    test_rows.append([sample, *(values / values.sum() if values.sum() > 0 else values)])
test_features = pd.DataFrame(test_rows, columns=["sample_id", *relative]).groupby("sample_id").median()
assert len(test_rows) == 35 and len(test_features) == 10 and set(test_features.index) == set(template.index)
scaler = StandardScaler().fit(soil)
reference = scaler.transform(soil)
for sample, query in zip(template.index, scaler.transform(test_features.loc[template.index])):
    distances = np.linalg.norm(reference - query, axis=1)
    nearest = soil.index[np.argsort(distances, kind="stable")[:3]]
    template.loc[sample] = labels.loc[nearest].mean()
destination = next(p for p in [Path("/kaggle/working"), Path("outputs"), Path(".")] if p.is_dir()) / "submission_3nn_goreli_doku.csv"
template.to_csv(destination, float_format="%.8f")
saved = pd.read_csv(destination).set_index("sample_id")
pd.testing.assert_frame_equal(saved, template, atol=1e-8, rtol=0)
assert saved.shape == (10, 11) and np.isfinite(saved).all().all()
assert np.all(np.diff(saved, axis=1) >= 0) and saved.ge(0).all().all() and saved.le(100).all().all() and saved["200"].eq(100).all()
print(destination, "· 10 numune, 11 geçerli yüzde")

outputs\submission_3nn_goreli_doku.csv · 10 numune, 11 geçerli yüzde
